This notebook is used to synthesize the trained models with HLS4MLs backend Vitis Unified. Some synthesis include bitfile-generation, and varies with strategy.

In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
from sklearn.metrics import accuracy_score

# Load Vitis into path
os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

In [4]:
model_to_test = 'pixsplit-hgq2'

model_configs = [
    {
        "description": "AdaptiveHP acc=0.7017 ebops=1150",
        "model_revision": "Training_AdaptiveHP",
        "keras_model_path": "pixsplit_hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7017_ebops=1150.keras",
        "hls4ml_strategy": "DA",
        "hls4ml_generate_bitfile": True,
        "hls4ml_revision": "VU_DA_bitfile",
    },
    {
        "description": "AdaptiveHP acc=0.7017 ebops=1150",
        "model_revision": "Training_AdaptiveHP",
        "keras_model_path": "pixsplit_hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7017_ebops=1150.keras",
        "hls4ml_strategy": "latency",
        "hls4ml_generate_bitfile": True,
        "hls4ml_revision": "VU_latency_bitfile",
    },
#    {
#        "description": "AdaptiveHP acc=0.7279 ebops=2449",
#        "model_revision": "Training_AdaptiveHP",
#        "keras_model_path": "pixsplit_hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7279_ebops=2449.keras",
#        "hls4ml_strategy": "DA",
#        "hls4ml_generate_bitfile": True,
#        "hls4ml_revision": "VU_DA_bitfile",
#    },
#    {
#        "description": "AdaptiveHP acc=0.7425 ebops=3899",
#        "model_revision": "Training_AdaptiveHP",
#        "keras_model_path": "pixsplit_hgq2/Training_AdaptiveHP/model_Training_AdaptiveHP_acc=0.7425_ebops=3899.keras",
#        "hls4ml_strategy": "DA",
#        "hls4ml_generate_bitfile": True,
#        "hls4ml_revision": "VU_DA_bitfile",
#    },
]

In [5]:
# Load dataset which is preprocessed in another notebook
X_train = np.load("Data/processed_data/X_train.npy")
X_val = np.load("Data/processed_data/X_val.npy")
X_test = np.load("Data/processed_data/X_test.npy")
y_train = np.load("Data/processed_data/y_train.npy")
y_val = np.load("Data/processed_data/y_val.npy")
y_test = np.load("Data/processed_data/y_test.npy")


FileNotFoundError: [Errno 2] No such file or directory: 'Data/processed_data/X_train.npy'

In [6]:
import os

def prepare_directory(model_config):
    output_dir = os.path.join(
        os.path.dirname(os.path.abspath(model_config["keras_model_path"])),
        f"hls4ml_prj_{model_config['hls4ml_revision']}",
    )
    os.makedirs(output_dir, exist_ok=True)

    description = f"""
    Description of HLS4ML-project.

    {model_config['description']}

    - Bitfile: {model_config['hls4ml_generate_bitfile']}
    - Environment: devenv-hgq+da (environment-HGQ+DA.yml)
    - Target Device: KV260 (xck26-sfvc784-2LV-c)
    - Dataset: Pixel Cluster Splitting
    - Vivado/Vitis: 2025.2
    - Model Architecture: {model_to_test}
    - Model Revision: {model_config['model_revision']}
    - HLS4ML Revision: {model_config['hls4ml_revision']}

    The model summary is in the parent-directory, `summary.txt`
    """
    with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
        f.write(description)

    return output_dir

In [7]:
from keras.models import load_model
import hgq.layers
import hls4ml

def compile_model(keras_model_path, output_dir, hls4ml_strategy):
    model = load_model(keras_model_path)
    
    hls_config = hls4ml.utils.config_from_keras_model(model, granularity='name')
    
    strategy = 'Distributed Arithmetic' if hls4ml_strategy == 'DA' else hls4ml_strategy
    hls_config['Model']['Strategy'] = strategy # https://fastmachinelearning.org/hls4ml/api/configuration.html#top-level-configuration

    hls_model = hls4ml.converters.convert_from_keras_model( 
        model,    
        backend     =   'vitisunified',
        hls_config  =   hls_config,
        output_dir  =   output_dir, 
        board       =   'kv260',
        part        =   'xck26-sfvc784-2LV-c',
        clock_period=   '5',
    )
    return hls_model.compile()

In [8]:
for model_config in model_configs:
    output_dir = prepare_directory(model_config)
    hls_model = compile_model(
        model_config["keras_model_path"],
        output_dir,
        model_config["hls4ml_strategy"],
    )
    # Create complete bitfile (Vitis Unified-backend) or IP-block (Vitis-backend)
    hls_model.build(
        synth=True,
        bitfile=model_config["hls4ml_generate_bitfile"],
        csim=False # CSIM and COSIM needs input_data_tb and output_data_tb https://fastmachinelearning.org/hls4ml/autodoc/hls4ml.converters.html#hls4ml.converters.convert_from_keras_model
    )

NotImplementedError: Heterogenous quantization for activations is only supported with IOType=io_parallel